In [1]:
import pandas as pd
import geopandas as gpd

# Load cleaned crime data
crime = pd.read_csv('../data/processed/crime_clean.csv')

# Load neighborhood boundaries
neighborhoods = gpd.read_file('../data/raw/neighborhoods.geojson')

print(crime.shape)
print(neighborhoods.shape)
neighborhoods.head()

(236447, 6)
(26, 9)


,OBJECTID,name,acres,neighborhood_id,sqmiles,Shape_Length,Shape_Area,shape_wkt,geometry
0,1,Roslindale,1605.568237,15,2.51,0.169120,0.000709,None,"MULTIPOLYGON (((-71.12593 42.272, -71.12611 42..."
1,2,Jamaica Plain,2519.245394,11,3.94,0.179578,0.001113,None,"POLYGON ((-71.10499 42.32609, -71.10503 42.326..."
2,3,Mission Hill,350.853564,13,0.55,0.059235,0.000155,None,"POLYGON ((-71.09043 42.33576, -71.0905 42.3357..."
3,4,Longwood,188.611947,28,0.29,0.038816,0.000083,None,"POLYGON ((-71.09811 42.33672, -71.09832 42.337..."
4,5,Bay Village,26.539839,33,0.04,0.015625,0.000012,None,"POLYGON ((-71.06663 42.34877, -71.06663 42.348..."


In [9]:
crime_geo = gpd.GeoDataFrame( crime, geometry=gpd.points_from_xy(crime['Long'], crime['Lat']))

In [10]:
print(type(crime_geo))
print(crime_geo.shape)
crime_geo.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
(236447, 7)


,OFFENSE_DESCRIPTION,DISTRICT,SHOOTING,OCCURRED_ON_DATE,Lat,Long,geometry
0,INVESTIGATE PERSON,B3,0,2023-01-27 22:44:00+00,42.271661,-71.099534,POINT (-71.09953 42.27166)
1,VERBAL DISPUTE,B2,0,2023-01-17 20:21:00+00,42.312597,-71.092875,POINT (-71.09288 42.3126)
2,INVESTIGATE PERSON,A1,0,2023-01-24 00:00:00+00,42.365700,-71.052892,POINT (-71.05289 42.3657)
3,INVESTIGATE PROPERTY,B3,0,2023-03-31 17:14:00+00,42.292788,-71.088519,POINT (-71.08852 42.29279)
4,ASSAULT - AGGRAVATED,B2,0,2023-01-26 09:00:00+00,42.310269,-71.089310,POINT (-71.08931 42.31027)


In [11]:
print(crime_geo.crs)
print(neighborhoods.crs)

None
EPSG:4326


In [12]:
crime_geo = crime_geo.set_crs(epsg=4326)
print(crime_geo.crs)

EPSG:4326


In [13]:
crime_with_neighborhoods = gpd.sjoin(
    crime_geo, 
    neighborhoods[['name', 'geometry']], 
    how='left', 
    predicate='within'
)

print(crime_with_neighborhoods.shape)
crime_with_neighborhoods.head()

(236448, 9)


,OFFENSE_DESCRIPTION,DISTRICT,SHOOTING,OCCURRED_ON_DATE,Lat,Long,geometry,index_right,name
0,INVESTIGATE PERSON,B3,0,2023-01-27 22:44:00+00,42.271661,-71.099534,POINT (-71.09953 42.27166),20.0,Mattapan
1,VERBAL DISPUTE,B2,0,2023-01-17 20:21:00+00,42.312597,-71.092875,POINT (-71.09288 42.3126),8.0,Roxbury
2,INVESTIGATE PERSON,A1,0,2023-01-24 00:00:00+00,42.365700,-71.052892,POINT (-71.05289 42.3657),7.0,North End
3,INVESTIGATE PROPERTY,B3,0,2023-03-31 17:14:00+00,42.292788,-71.088519,POINT (-71.08852 42.29279),21.0,Dorchester
4,ASSAULT - AGGRAVATED,B2,0,2023-01-26 09:00:00+00,42.310269,-71.089310,POINT (-71.08931 42.31027),8.0,Roxbury


In [14]:
print(neighborhoods.total_bounds)
print(crime_geo.total_bounds)

[-71.1912491   42.22791113 -70.92277988  42.3969775 ]
[-71.22812796  42.17145601 -70.83688105  42.45998696]


In [15]:
crime_with_neighborhoods = gpd.sjoin(
    crime_geo,
    neighborhoods[['name', 'geometry']],
    how='left',
    predicate='within'
)

print(crime_with_neighborhoods.shape)
crime_with_neighborhoods['name'].value_counts()



(236448, 9)


name
Dorchester                 54640
Roxbury                    28153
South End                  15430
South Boston               14064
Downtown                   13603
Jamaica Plain              13448
East Boston                12496
Brighton                   11415
Hyde Park                  10070
Back Bay                    9813
Mattapan                    8193
West Roxbury                7823
Fenway                      5658
Allston                     5407
Roslindale                  4955
West End                    4103
Charlestown                 3948
Mission Hill                3348
Beacon Hill                 2604
North End                   1971
South Boston Waterfront     1904
Chinatown                   1513
Longwood                    1117
Bay Village                  356
Leather District             313
Harbor Islands                11
Name: count, dtype: int64

In [16]:
crime_neighborhoods = crime_with_neighborhoods[['OFFENSE_DESCRIPTION', 'SHOOTING', 'OCCURRED_ON_DATE', 'Lat', 'Long', 'name']].copy()
crime_neighborhoods = crime_neighborhoods.rename(columns={'name': 'neighborhood'})
crime_neighborhoods.head()

,OFFENSE_DESCRIPTION,SHOOTING,OCCURRED_ON_DATE,Lat,Long,neighborhood
0,INVESTIGATE PERSON,0,2023-01-27 22:44:00+00,42.271661,-71.099534,Mattapan
1,VERBAL DISPUTE,0,2023-01-17 20:21:00+00,42.312597,-71.092875,Roxbury
2,INVESTIGATE PERSON,0,2023-01-24 00:00:00+00,42.365700,-71.052892,North End
3,INVESTIGATE PROPERTY,0,2023-03-31 17:14:00+00,42.292788,-71.088519,Dorchester
4,ASSAULT - AGGRAVATED,0,2023-01-26 09:00:00+00,42.310269,-71.089310,Roxbury


In [17]:
crime_neighborhoods.to_csv('../data/processed/crime_with_neighborhoods.csv', index=False)

In [18]:
mbta = gpd.read_file('../data/raw/MBTA_NODE.shp')
print(mbta.shape)
print(mbta.columns.tolist())
mbta.head()

(170, 5)
['STATION', 'LINE', 'TERMINUS', 'ROUTE', 'geometry']


,STATION,LINE,TERMINUS,ROUTE,geometry
0,Park Street,GREEN/RED,N,GREEN B C D E / RED A - Ashmont B - Braintree...,POINT (236064.005 900737.761)
1,JFK/UMass,RED,N,A - Ashmont B - Braintree C - Alewife,POINT (236899.089 896780.048)
2,State,BLUE/ORANGE,N,BLUE Bowdoin to Wonderland / ORANGE Forest Hil...,POINT (236427.189 901016.63)
3,Roxbury Crossing,ORANGE,N,Forest Hills to Oak Grove,POINT (233318.703 897936.59)
4,Airport Terminal B2,SILVER,N,SL1,POINT (239674.505 901509.172)


In [19]:
print(mbta.crs)

EPSG:26986


In [20]:
mbta = mbta.to_crs(epsg=4326)
print(mbta.crs)
mbta.head()

EPSG:4326


,STATION,LINE,TERMINUS,ROUTE,geometry
0,Park Street,GREEN/RED,N,GREEN B C D E / RED A - Ashmont B - Braintree...,POINT (-71.06224 42.35631)
1,JFK/UMass,RED,N,A - Ashmont B - Braintree C - Alewife,POINT (-71.05236 42.32064)
2,State,BLUE/ORANGE,N,BLUE Bowdoin to Wonderland / ORANGE Forest Hil...,POINT (-71.05782 42.3588)
3,Roxbury Crossing,ORANGE,N,Forest Hills to Oak Grove,POINT (-71.09573 42.33121)
4,Airport Terminal B2,SILVER,N,SL1,POINT (-71.01837 42.36308)


In [21]:
neighborhoods_proj = neighborhoods.to_crs(epsg=26986)
mbta_proj = mbta.to_crs(epsg=26986)

print(neighborhoods_proj.crs)
print(mbta_proj.crs)

EPSG:26986
EPSG:26986


In [22]:
neighborhoods_proj['centroid'] = neighborhoods_proj.geometry.centroid
print(neighborhoods_proj[['name', 'centroid']].head())

            name                       centroid
0     Roslindale  POINT (230792.997 892516.277)
1  Jamaica Plain  POINT (231734.222 895324.512)
2   Mission Hill  POINT (232751.627 897992.054)
3       Longwood  POINT (232542.397 898753.889)
4    Bay Village  POINT (235508.961 899933.943)


In [23]:
def count_nearby_stops(centroid, stops, radius=1609):
    return sum(stops.distance(centroid) <= radius)

neighborhoods_proj['transit_score'] = neighborhoods_proj['centroid'].apply(
    lambda c: count_nearby_stops(c, mbta_proj.geometry)
)

neighborhoods_proj[['name', 'transit_score']].sort_values('transit_score', ascending=False)

,name,transit_score
4,Bay Village,37
6,Chinatown,33
9,South End,32
5,Leather District,30
14,Beacon Hill,29
10,Back Bay,29
15,Downtown,29
3,Longwood,24
16,Fenway,21
13,West End,19


In [24]:
def count_nearby_stops_boundary(neighborhood_geom, stops, radius=1609):
    return sum(stops.distance(neighborhood_geom) <= radius)

neighborhoods_proj['transit_score'] = neighborhoods_proj.geometry.apply(
    lambda geom: count_nearby_stops_boundary(geom, mbta_proj.geometry)
)

neighborhoods_proj[['name', 'transit_score']].sort_values('transit_score', ascending=False)

,name,transit_score
10,Back Bay,57
23,South Boston,55
16,Fenway,53
9,South End,51
15,Downtown,47
8,Roxbury,45
22,South Boston Waterfront,44
2,Mission Hill,43
6,Chinatown,43
4,Bay Village,41


In [25]:
from shapely.geometry import Point

# BU campus center point
bu_point = gpd.GeoDataFrame(
    geometry=[Point(-71.1054, 42.3505)], 
    crs='EPSG:4326'
).to_crs(epsg=26986).geometry[0]

# Distance from each neighborhood centroid to BU in miles
neighborhoods_proj['bu_distance_miles'] = neighborhoods_proj['centroid'].apply(
    lambda c: c.distance(bu_point) / 1609
).round(2)

neighborhoods_proj[['name', 'transit_score', 'bu_distance_miles']].sort_values('bu_distance_miles')

,name,transit_score,bu_distance_miles
16,Fenway,53,0.62
3,Longwood,38,0.82
10,Back Bay,57,1.27
2,Mission Hill,43,1.30
24,Allston,35,1.31
9,South End,51,1.85
4,Bay Village,41,1.86
14,Beacon Hill,41,1.95
6,Chinatown,43,2.24
13,West End,29,2.26


In [26]:
transit_bu_scores = neighborhoods_proj[['name', 'transit_score', 'bu_distance_miles']].copy()
transit_bu_scores.to_csv('../data/processed/transit_bu_scores.csv', index=False)
print("Saved!")

Saved!
